In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sbs
import math as math
import os
from os import listdir
pd.options.mode.use_inf_as_na = True
import tkinter
from tkinter import filedialog
import scipy as scp

In [12]:
import os
import pandas as pd

def data_to_dataframe(pathinfo, y_col='Y (cm)', x_col='X (cm)', speed_col='SPEED#wcentroid (cm/s)', vy_col='VY (cm/s)', frame_col='frame'):
    """
    Reads CSV files from a specified directory tree, extracts specific columns, and compiles them into a single pandas DataFrame.

    This function traverses the directory tree starting at `pathinfo`, identifies CSV files with "Trial" in their filenames,
    and extracts specific columns. It combines these extracted columns into a single DataFrame with additional metadata 
    about the trial and condition.

    Parameters:
    -----------
    pathinfo : str
        The root directory from which to start searching for CSV files.
    y_col : str, optional
        Column name for Y coordinates, default is 'Y (cm)'.
    x_col : str, optional
        Column name for X coordinates, default is 'X (cm)'.
    speed_col : str, optional
        Column name for speed, default is 'SPEED#wcentroid (cm/s)'.
    vy_col : str, optional
        Column name for Y velocity, default is 'VY (cm/s)'.
    frame_col : str, optional
        Column name for frame, default is 'frame'.

    Returns:
    --------
    pandas.DataFrame
        A DataFrame containing concatenated data from all relevant CSV files, with columns for Y, X, Speed, VY, Frame, Trial, and Condition.

    Notes:
    ------
    - The function assumes that the CSV files contain the specified columns.
    - The 'Trial' and 'Condition' columns are derived from the directory structure where the CSV files are found.
    - Uses os.path.join for cross-platform compatibility.

    Example:
    --------
    >>> df = data_to_dataframe("C:\\data\\experiments")
    >>> print(df.head())
    """
    
    df_files = []
    i = 0

    for root, dirs, files in os.walk(pathinfo):
        for f in files:
            if f.endswith(".csv") and "Trial" in f:
                file_path = os.path.join(root, f)
                try:
                    a = pd.read_csv(file_path)
                    data = {
                        "Y": a[y_col],
                        "X": a[x_col],
                        "Speed": a[speed_col],
                        "VY": a[vy_col],
                        "Frame": a[frame_col],
                        "Trial": os.path.basename(root),
                        "Condition": os.path.basename(os.path.dirname(root))
                    }
                    df_files.append(pd.DataFrame(data))
                    i += 1
                except KeyError as e:
                    print(f"Missing column in file {file_path}: {e}")
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

    if df_files:
        complete_df = pd.concat(df_files, ignore_index=True)
    else:
        complete_df = pd.DataFrame()

    return complete_df

def clean_dataframe(df):
    """
    Cleans the DataFrame by dropping rows with missing values and filtering 
    rows based on the 'Speed' column.

    This function removes rows with any missing values and retains only rows 
    where the 'Speed' value is between 0.5 and 2.0 (exclusive).

    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame to be cleaned.

    Returns:
    --------
    pandas.DataFrame
        The cleaned DataFrame with no missing values and 'Speed' values 
        between 0.5 and 2.0.
    
    Example:
    --------
    >>> df = data_to_dataframe("C:\\data\\experiments")
    >>> clean_df = clean_dataframe(df)
    >>> print(clean_df.head())
    """

    df_cleaned = df.dropna()
    df_cleaned = df_cleaned[df_cleaned["Speed"].between(0.5, 2.0, inclusive='neither')]
    return df_cleaned

def calculate_distance_from_fixed_point(df):
    """
    Calculates distance from a fixed point ('X_1', 'Y_1') to each row's ('X', 'Y') coordinates in the DataFrame.

    This function iterates over each row in the input DataFrame and calculates the Euclidean distance 
    from a fixed point ('X_1', 'Y_1') to each row's ('X', 'Y') coordinates.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame containing 'X', 'Y', 'Frame', 'Trial', and 'Condition' columns.

    Returns:
    --------
    pandas.DataFrame
        DataFrame with additional column 'Distance' appended to the input DataFrame.

    Example:
    --------
    >>> df = pd.DataFrame({
    >>>     'X': [1, 2, 3],
    >>>     'Y': [4, 5, 6],
    >>>     'Frame': [1, 2, 3],
    >>>     'Trial': ['Trial1', 'Trial1', 'Trial2'],
    >>>     'Condition': ['Condition1', 'Condition1', 'Condition2']
    >>> })
    >>> df_processed = calculate_distance_from_fixed_point(df)
    >>> print(df_processed.head())
    """

    # Fixed point coordinates
    x_1 = 14
    y_1 = 14

    # Calculate distance for each row
    df['Distance'] = np.nan
    for idx, row in df.iterrows():
        x_2 = row['X']
        y_2 = row['Y']
        dist = math.sqrt((x_2 - x_1)**2 + (y_2 - y_1)**2)
        df.loc[idx, 'Distance'] = dist

    return df




In [9]:
tkinter.Tk().withdraw()
filepath = filedialog.askdirectory()
primary_df = data_to_dataframe(filepath)

,Y,X,Speed,VY,Frame,Trial,Condition
0,17.353,27.751,0.000,0.000,0.0,Trial 1,10-2 cVA in Ethanol
1,16.454,27.489,10.801,-8.995,1.0,Trial 1,10-2 cVA in Ethanol
2,15.785,27.498,7.246,-6.690,2.0,Trial 1,10-2 cVA in Ethanol
3,15.362,27.081,5.449,-4.234,3.0,Trial 1,10-2 cVA in Ethanol
4,15.068,27.086,0.151,-2.931,4.0,Trial 1,10-2 cVA in Ethanol
